# colab_03_bandit_rl — cost-aware contextual bandit on FREE T4 (RQ2, core novelty)

Gate: DECISION_01 + DECISION_02. One training run, sweep `lambda` (cost of failure) post-hoc — no re-training loops.

Outputs: `bandit_head.pt` + `DECISION_03.json` (freezes best lambda on CALIB only).

In [ ]:
# 0. Setup — self-recovering: finds repo, unzips upload, or clones (public or private via token)
import os, sys, json, hashlib, subprocess, zipfile
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE = Path('/content/drive/MyDrive/Research/colab_ckpt')
    DRIVE.mkdir(parents=True, exist_ok=True)
    print('Drive:', DRIVE)
except Exception as e:
    print('No Colab Drive (local run?):', e)
    DRIVE = Path('./colab_ckpt')
    DRIVE.mkdir(parents=True, exist_ok=True)

def _find_backend():
    """Return BACKEND dir (containing src/gateway/service.py) or None."""
    cands = [Path('/content/code-mixed-gateway/backend'), Path('/content/Research/backend'),
             Path('/content/backend'), Path.cwd()/'backend', Path.cwd()/'code-mixed-gateway'/'backend']
    try:
        for p in Path('/content').iterdir():
            if p.is_dir():
                cands += [p, p/'backend']
    except Exception: pass
    for c in cands:
        if (c/'src'/'gateway'/'service.py').exists(): return c
    # deep search one level for upload_tmp style dirs
    try:
        for p in Path('/content').glob('*/**/service.py'):
            if p.parent.parent.name == 'gateway' and p.parent.name == 'gateway':
                be = p.parent.parent.parent  # .../src/gateway/service.py -> backend?
                # walk up to dir containing 'src'
                q = p.parent
                while q != Path('/content') and q.name != 'src': q = q.parent
                be = q.parent
                if (be/'src'/'gateway'/'service.py').exists(): return be
    except Exception: pass
    return None

def _try_unzip_uploads():
    roots = [Path('/content')]
    try: roots.append(DRIVE)
    except Exception: pass
    for root in roots:
        try: zips = list(root.glob('backend*.zip'))
        except Exception: continue
        for z in zips:
            dest = Path('/content/repo_upload')
            if (dest/'backend'/'src'/'gateway'/'service.py').exists(): return True
            print(f'unzipping {z} -> {dest} ...')
            dest.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(z) as zf: zf.extractall(dest)
            # handle zip containing backend/ at top or nested one level
            if not (dest/'backend'/'src'/'gateway'/'service.py').exists():
                for sub in dest.iterdir():
                    if (sub/'backend'/'src'/'gateway'/'service.py').exists():
                        import shutil
                        shutil.move(str(sub/'backend'), str(dest/'backend_tmp'))
                        shutil.move(str(dest/'backend_tmp'), str(dest/'backend'))
                        break
            print('unzip done')
            return True
    return False

def _try_clone():
    dest = Path('/content/code-mixed-gateway')
    if dest.exists(): return dest.exists()
    token = ''
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN') or ''
    except Exception: token = os.environ.get('GITHUB_TOKEN','') or ''
    url = ('https://'+token+'@github.com/tusharsaharan/code-mixed-gateway.git') if token else 'https://github.com/tusharsaharan/code-mixed-gateway.git'
    print('cloning (token=%s) ...' % ('yes' if token else 'no'))
    r = subprocess.run(['git','clone',url,str(dest)], capture_output=True, text=True)
    print('clone rc=', r.returncode)
    if r.returncode != 0: print((r.stderr or '')[-500:])
    return dest.exists()

BACKEND = _find_backend()
if BACKEND is None and _try_unzip_uploads(): BACKEND = _find_backend()
if BACKEND is None and _try_clone(): BACKEND = _find_backend()
print('BACKEND =', BACKEND)
assert BACKEND is not None, ('REPO NOT FOUND. Fastest fix (2 min): on your PC zip the backend folder '
    '(Research/backend -> backend.zip), drag backend.zip into Colab Files panel (folder icon, left), then '
    'Cell > Run this cell again. Alternative: make the GitHub repo public, or add a GITHUB_TOKEN secret.')
REPO = BACKEND.parent
DATA = BACKEND/'data'
print('REPO =', REPO, '| DATA exists:', DATA.exists())

%pip install -q pydantic pydantic-settings numpy httpx 2>&1 | tail -1
src = str(BACKEND/'src')
if src not in sys.path: sys.path.insert(0, src)
import gateway
print('import gateway OK from', gateway.__file__)


In [ ]:
# 1. Offline bandit data: context=MiniLM+heuristic (reuse NB02 cache if present) else rebuild quickly
import json, numpy as np
from sentence_transformers import SentenceTransformer
rows = [json.loads(l) for l in open(DRIVE/'calibration_real.jsonl', encoding='utf-8')]
lab = {json.loads(l)['id']: json.loads(l) for l in open(DRIVE/'groq_labels.jsonl', encoding='utf-8')}
splits = json.loads((DRIVE/'splits.json').read_text())
from gateway.modules.m4_router.difficulty import DifficultyScorer
scorer = DifficultyScorer()
enc = SentenceTransformer('all-MiniLM-L6-v2')
ids = [r['id'] for r in rows]
E = enc.encode([lab[i]['text'] for i in ids], batch_size=64, show_progress_bar=True, normalize_embeddings=True)
X = {}
for k, rid in enumerate(ids):
    f = scorer.features(lab[rid]['text'])
    h = np.array([f.code_mix_ratio, f.entity_density, min(1,f.math_marker_count/3), min(1,f.char_count/400)], dtype=np.float32)
    X[rid] = np.concatenate([h, E[k]]).astype(np.float32)
# reward: cheap success saves cost, cheap fail pays lambda. premium always 0 (safe, costly).
# proxy costs (free tiers ~0, so use paper rates for optimization): cheap=0.06/1M, premium=2.50/1M per pricing.py
FAIL = {r['id']: (0 if r['cheap_success'] else 1) for r in rows}
TOK = {rid: max(10, len(lab[rid]['text'].split())*2) for rid in ids}  # approx tokens
print('contexts:', len(X), 'dim:', len(next(iter(X.values()))))

In [ ]:
# 2. Baselines: LinUCB-lite (numpy) + policy head (torch, T4, head-only, frozen encoder)
import torch, torch.nn as nn, random
import numpy as np
device = 'cuda' if torch.cuda.is_available() else 'cpu'
D = len(next(iter(X.values())))
class Head(nn.Module):
    def __init__(self, d): super().__init__(); self.net = nn.Sequential(nn.Linear(d,128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128,2))
    def forward(self, x): return self.net(x)
head = Head(D).to(device)
opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
tr = splits['train']
Xtr = torch.tensor(np.stack([X[i] for i in tr]), dtype=torch.float32)
# REINFORCE-with-baseline on offline both-outcome data: for each x we know fail (0/1).
# loss = -E_a~pi[ R(x,a) ] + CQL penalty. R(cheap)= save - lam*fail, R(prem)=0.
SAVE = 0.7  # normalized saving unit; lambda swept post-hoc, train once with lam=2
LAM_TRAIN = 2.0
ytr = torch.tensor([FAIL[i] for i in tr], dtype=torch.float32)
save = torch.tensor([SAVE]*len(tr))
ds = torch.utils.data.TensorDataset(Xtr, ytr, save)
ld = torch.utils.data.DataLoader(ds, batch_size=64, shuffle=True)
head.train()
for ep in range(20):
    tot = 0
    for xb, yb, sv in ld:
        xb, yb, sv = xb.to(device), yb.to(device), sv.to(device)
        logits = head(xb)
        logp = torch.log_softmax(logits, dim=1)
        R_cheap = sv - LAM_TRAIN*yb; R_prem = torch.zeros_like(R_cheap)
        R = torch.stack([R_cheap, R_prem], dim=1)
        baseline = R.mean(dim=1, keepdim=True)
        pg = -(torch.softmax(logits, dim=1) * (R - baseline) * logp).sum(dim=1).mean()
        cql = (torch.logsumexp(logits, dim=1) - logits.mean(dim=1)).mean() * 0.1
        loss = pg + cql
        opt.zero_grad(); loss.backward(); opt.step(); tot += float(loss)
    if ep % 5 == 0: print(f'ep{ep} loss {tot/len(ld):.4f}')
torch.save(head.state_dict(), DRIVE/'bandit_head.pt')
print('saved bandit_head.pt')

In [ ]:
# 3. Post-hoc lambda sweep on CALIB only (no retrain). Freeze best lambda.
import torch, numpy as np, json
head.load_state_dict(torch.load(DRIVE/'bandit_head.pt', map_location=device)); head.eval()
ca = splits['calib']
Xca = torch.tensor(np.stack([X[i] for i in ca]), dtype=torch.float32).to(device)
with torch.no_grad(): p_fail_bandit = torch.softmax(head(Xca), dim=1)[:,0].cpu().numpy()  # prob route cheap? use as score
# Actually use P(fail) estimate: reuse NB02 winner blended 0.5/0.5 for stability
import pickle
try:
    lr = pickle.load(open(DRIVE/'logreg.pkl','rb'))
    p_lr = lr.predict_proba(np.stack([X[i][:389] if len(X[i])>=389 else X[i] for i in ca]))[:,1]
except Exception as e:
    print('logreg reuse failed, bandit only:', e); p_lr = p_fail_bandit
score = 0.5*p_fail_bandit + 0.5*p_lr
yca = np.array([FAIL[i] for i in ca])
rows_out = []
for lam in [0.5, 1.0, 2.0, 5.0]:
    # threshold chosen to maximize reward on calib for this lambda (report; NB04 recalibrates conformally)
    best = max((float(((score<t).astype(float)*(SAVE - lam*yca)).mean()), float(t)) for t in [i/200 for i in range(201)])
    rows_out.append({'lambda': lam, 'thresh_reward_opt': round(best[1],4), 'mean_reward': round(best[0],4)})
    print(rows_out[-1])
(DRIVE/'DECISION_03.json').write_text(json.dumps({'stage':'03_bandit','lambdas':rows_out,'frozen_head':'bandit_head.pt','note':'NB04 picks conformal tau, not these reward-opt thresholds'}, indent=2))